# E213 – Time Series and Seasonality Demo (Bloomy Days Example)

1. Building a simple **monthly time series** for a business.
2. Fitting a **linear trend model**.
3. Computing **seasonal indices** from historical data.
4. Creating **seasonally adjusted forecasts** for the next 12 months.

We will use a fictional online flower shop called **Bloomy Days** to make the ideas concrete and easy to follow.


## 1. Business Scenario: Bloomy Days

You manage **Bloomy Days**, an online flower shop that has been operating for **3 years**.

- You have **monthly order data** from **January 2020 to December 2022** (36 months).
- Orders seem to **increase over time** as the business grows (a **trend**).
- There is clear **seasonality**:
  - **February** (Valentine’s Day) is strong.
  - **May–July** (Mother’s Day + wedding season) are very strong.
  - Some months (like November) are weaker.

In this notebook, we will **simulate** that data (instead of reading a real CSV file), and then treat it as if it were real historical data.

Our goals:

1. Fit a **time trend** to the historical data.
2. Forecast **baseline monthly orders** for the next year (2023), **without seasonality**.
3. Compute **seasonal indices** from the 3 years of data.
4. Combine the two to create **final, seasonally adjusted monthly forecasts** for 2023.


## 2. Setup: Import Libraries

We will use:

- `pandas` for working with time series tables.
- `numpy` for numerical work and random number generation.
- `statsmodels` for fitting a simple linear regression trend model.


In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# For reproducible random numbers
np.random.seed(42)

ModuleNotFoundError: No module named 'statsmodels'

## 3. Create a 3-Year Monthly Dataset with Seasonality

Instead of reading in a file, we will generate a **synthetic** dataset that behaves like a real business.

### Steps:

1. Create a **monthly date range** from Jan 2020 to Dec 2022 (36 months).
2. Generate a **baseline level** of orders (around 500 per month) with some randomness.
3. Define a **seasonal pattern**: 12 multipliers, one per calendar month (Jan–Dec), repeated for 3 years.
4. Multiply the baseline orders by the seasonal pattern to get the final `Orders`.

> Think of the seasonal multipliers as “how strong this month is compared to an average month.”  
> - Multiplier > 1.0 → above-average month (more orders)  
> - Multiplier < 1.0 → below-average month (fewer orders)


In [ ]:
# 36 months of data: Jan 2020 – Dec 2022
dates = pd.date_range(start="2020-01-01", periods=36, freq="M")

# Baseline orders around 500, with some noise
base_orders = np.random.normal(loc=500, scale=40, size=36)

# Seasonal pattern for each month (Jan to Dec)
# Valentine’s Day (Feb) and wedding/Mother's Day season (May–Jul) are strong.
seasonal_pattern_one_year = [
    0.80,  # Jan  - slightly below average
    1.20,  # Feb  - Valentine's Day spike
    0.95,  # Mar
    1.05,  # Apr
    1.40,  # May  - Mother's Day
    1.60,  # Jun  - weddings
    1.50,  # Jul  - weddings
    1.10,  # Aug
    0.95,  # Sep
    0.90,  # Oct
    0.85,  # Nov  - slower month
    0.90   # Dec  - some holidays, but assume not as big as others for this shop
]

# Repeat the 12-month pattern 3 times for 3 years
seasonal_pattern = seasonal_pattern_one_year * 3

# Apply seasonality to the baseline
orders = base_orders * seasonal_pattern

# Build the DataFrame
df = pd.DataFrame({
    "Month": dates,
    "Orders": orders
})

df

The table above shows the first few months of our simulated data.

- `Month` is the end-of-month date for each month.
- `Orders` is the number of flower orders Bloomy Days received that month (after adding noise and seasonality).

We will treat this as our **historical time series**.


## 4. Add a Time Index for the Trend Model

To fit a simple linear trend, we will add a Time_Index column:

- Time_Index = 1 for the first month (Jan 2020)
- Time_Index = 2 for Feb 2020
- ...
- Time_Index = 36 for Dec 2022

Our trend model will look like:

Orders_t = β0 + β1 * Time_Index_t + ε_t

Orders_t = β0 + β1 * Time_Index_t + ε_t

Where:
β0 = intercept
β1 = average monthly change
ε_t = random error

In [ ]:
# Add a simple time index: 1, 2, ..., 36
df["Time_Index"] = np.arange(1, len(df) + 1)

df

## 5. Fit a Linear Trend Model (Orders ~ Time_Index)

We now fit a simple **Ordinary Least Squares (OLS)** regression model using `statsmodels`:

```text
Orders = β0 + β1 * Time_Index + error
```

### Interpretation:

- If β1 > 0, orders are increasing over time (upward trend).
- The model **does not include seasonality yet**. It just captures the underlying growth pattern over the 3 years.


In [ ]:
# Fit a linear trend: Orders ~ Time_Index
trend_model = smf.ols(formula="Orders ~ Time_Index", data=df).fit()

trend_model.summary()

Look at the regression output:

- The **coefficient for `Time_Index`** (β₁) tells you the **average change in orders per month**.
- The **Intercept** (β₀) is the model’s estimate of orders when `Time_Index = 0` (not directly meaningful here but needed for the equation).
- For forecasting, we mostly care that the model captures a reasonable **upward or downward trend**.

Next, we will use this model to forecast the **baseline** number of orders for the next 12 months, **ignoring seasonality**.


## 6. Create a Baseline Forecast for the Next Year (No Seasonality Yet)

We want to forecast orders for **Jan–Dec 2023**.

Steps:

1. Create a new DataFrame `df_forecast` with 12 new months (2023-01 to 2023-12).
2. Assign **Time_Index values 37–48** for these months (continuing from our 36 historical months).
3. Use the fitted `trend_model` to predict `Forecasted_Orders` for each of the 12 months.

> These forecasts will contain **trend only**, not seasonality.


In [ ]:
# Create a forecast DataFrame for the next 12 months after the historical period
forecast_dates = pd.date_range(start="2023-01-01", periods=12, freq="M")

df_forecast = pd.DataFrame({
    "Month": forecast_dates,
    "Time_Index": np.arange(len(df) + 1, len(df) + 13)  # 37 to 48
})

# Predict baseline (trend-only) orders
df_forecast["Forecasted_Orders"] = trend_model.predict(df_forecast)

df_forecast

## 7. Compute Seasonal Indices from Historical Data

Now we want to **quantify** the seasonal pattern using the historical data.

### Idea:

1. For each **calendar month** (Jan, Feb, ..., Dec), compute the **average Orders** across the 3 years.
2. Compute the **overall average** Orders across all 36 months.
3. For each month, compute a **Seasonal Index**:

\[ \text{Seasonal Index for month m} = \frac{\text{Average Orders for month m}}{\text{Overall Average Orders}} \]

### Interpretation:

- Seasonal Index = 1.40 → this month tends to be **40% higher** than an average month.
- Seasonal Index = 0.80 → this month tends to be **20% lower** than an average month.

We will use `Month.dt.month` (1–12) to group by month across years.


In [ ]:
# 1. Average orders by calendar month (1=Jan, 2=Feb, ..., 12=Dec)
monthly_orders = df["Orders"].groupby(df["Month"].dt.month).mean()
monthly_orders

In [ ]:
# 2. Overall average across all months
overall_avg_orders = df["Orders"].mean()
overall_avg_orders

In [ ]:
# 3. Seasonal index for each month = (month-average / overall average)
seasonal_index = monthly_orders / overall_avg_orders
seasonal_index

The table above shows the **seasonal index** for each month (1–12).

- Values **greater than 1** indicate months that are typically **above average** (e.g., Valentine’s Day or wedding season).
- Values **less than 1** indicate months that are typically **below average**.

We will now apply these seasonal indices to our **2023 baseline forecasts**.


## 8. Attach Seasonal Indices to the Forecast Months

Each row in `df_forecast` represents a future month in 2023.

Steps:

1. Extract the **calendar month number** from `Month` (1–12) into a new column `Month_Index`.
2. Use that number to **look up** the corresponding seasonal index (from `seasonal_index` above).
3. Store that as a new column `Seasonal_Index` in `df_forecast`.

This connects each forecasted period (e.g., Feb 2023) with the appropriate seasonal factor for that month (e.g., February’s index).


In [ ]:
# Extract month number (1–12) for each forecasted month
df_forecast["Month_Index"] = df_forecast["Month"].dt.month

# Map Month_Index -> Seasonal_Index using our previously computed seasonal_index series
df_forecast["Seasonal_Index"] = df_forecast["Month_Index"].apply(lambda m: seasonal_index[m])

df_forecast

## 9. Create Seasonally Adjusted Forecasts

Now we combine **trend** and **seasonality**:

Seasonally Adjusted Forecast = Baseline Trend Forecast * Seasonal Index

For each month in 2023:

1. Take the `Forecasted_Orders` (trend only).
2. Multiply by the `Seasonal_Index` for that month.
3. Store the result in a new column `Seasonally_Adjusted_Forecast`.

This gives us a more realistic forecast that includes both **growth over time** and **seasonal ups and downs**.


In [ ]:
# Multiply trend forecast by seasonal index to get final forecast
df_forecast["Seasonally_Adjusted_Forecast"] = (
    df_forecast["Forecasted_Orders"] * df_forecast["Seasonal_Index"]
)

df_forecast[["Month",
             "Forecasted_Orders",
             "Seasonal_Index",
             "Seasonally_Adjusted_Forecast"]]

## 10. Wrap-Up and Discussion

You now have:

- A **historical time series** of monthly orders for Bloomy Days (2020–2022).
- A **trend model** that captures the long-term growth in orders.
- **Seasonal indices** that quantify how strong each month is relative to an average month.
- **Seasonally adjusted forecasts** for each month of 2023.

### Questions to Ponder

1. **Interpretation of Seasonal Index**  
   - If June has a seasonal index of 1.6, what does that tell you about June compared to an average month?  
2. **Managerial Use**  
   - How could Bloomy Days use these forecasts to plan staffing, inventory, or marketing campaigns?  
3. **Deseasonalizing** (Optional)  
   - If you wanted to remove seasonality from the historical data, how would you do it?  
     (Hint: divide each actual value by the seasonal index for that month.)
